# Week 2: Data Collection and Preprocessing Framework (Python)

**Project:** E-Governance Service Delivery Analytics

This notebook is a working prototype of the framework described in the Week 2 DOC file. It simulates the full path from raw public data to an analysis-ready dataset:

1. Collect raw files and record **metadata** (source log, row counts, checksums)
2. **Parse and standardise** (column names, text, dates, numbers, state names)
3. **Validate** with a rule set (before cleaning)
4. **Clean** (units, invalid values, duplicates, outliers, missing values)
5. **Integrate** two sources (service performance + digital readiness)
6. **Transform** (features, scaling, encoding)
7. **Validate again** (after cleaning) and produce a quality report and charts

> **Important:** the raw files here are **simulated** to look like messy public data (mixed formats, name variants, NA tokens, duplicates, wrong units). They are not real government statistics. To use real data, replace the two files in `data/raw/` with real extracts that have the same columns.

## 1. Setup

In [ ]:
import os, re, json, hashlib, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer

sns.set_theme(style="whitegrid")
for d in ["data/raw", "data/interim", "data/processed", "charts", "logs"]:
    os.makedirs(d, exist_ok=True)

STATES = ["Uttar Pradesh", "Maharashtra", "Tamil Nadu", "Karnataka", "Gujarat", "Rajasthan",
          "West Bengal", "Bihar", "Kerala", "Madhya Pradesh", "Odisha", "Telangana"]
VARIANTS = {  # messy spellings seen in real public data
    "Uttar Pradesh": ["UTTAR PRADESH", "U.P.", "Uttar Pradesh "],
    "Maharashtra": ["MAHARASHTRA", "maharashtra"],
    "Tamil Nadu": ["TAMIL NADU", "Tamilnadu"],
    "Karnataka": ["KARNATAKA"],
    "Gujarat": ["GUJARAT", "Gujrat"],
    "Rajasthan": ["RAJASTHAN"],
    "West Bengal": ["WEST BENGAL", "W.B.", "West Bengal "],
    "Bihar": ["BIHAR"],
    "Kerala": ["KERALA", "kerala"],
    "Madhya Pradesh": ["MADHYA PRADESH", "M.P.", "Madhya  Pradesh"],
    "Odisha": ["Orissa", "ODISHA"],
    "Telangana": ["TELANGANA", "Telengana"],
}
def norm_label(s):
    """Normalise a label: lower-case, remove dots, collapse spaces."""
    return re.sub(r"\s+", " ", re.sub(r"\.", "", str(s)).strip().lower())

STATE_MAP = {norm_label(v): std for std, vs in VARIANTS.items() for v in [std] + vs}

## 2. Collect raw data (simulated) and log metadata

In a real project this step downloads files (CSV, XLSX, JSON, PDF tables) from public portals. Here two raw files are simulated:

- **Source A**: service performance (state, service, date, requests, processing time, resolution, rating)
- **Source B**: digital readiness (internet subscribers per 100 people, digital literacy %)

Raw files are saved **unchanged** and never edited. Every file gets a metadata record (name, source, date accessed, rows, columns, checksum).

In [ ]:
def simulate_source_a(seed=7):
    rng = np.random.default_rng(seed)
    params = {"Uttar Pradesh": (2.0, 0.52), "Maharashtra": (1.8, 0.72), "Tamil Nadu": (1.4, 0.76),
              "Karnataka": (1.3, 0.74), "Gujarat": (1.2, 0.70), "Rajasthan": (1.1, 0.55),
              "West Bengal": (1.3, 0.58), "Bihar": (1.4, 0.45), "Kerala": (1.0, 0.85),
              "Madhya Pradesh": (1.2, 0.50), "Odisha": (1.0, 0.50), "Telangana": (1.2, 0.73)}
    services = {"Birth Certificate": (6, 1.2), "Income Certificate": (12, 1.0),
                "Land Records": (18, 0.6), "Driving Licence": (9, 0.8)}
    rows = []
    for m in range(1, 13):
        month_end = pd.Timestamp(2025, m, 1) + pd.offsets.MonthEnd(0)
        for st, (w, base) in params.items():
            adoption = float(np.clip(base + 0.012 * (m - 1) + rng.normal(0, 0.01), 0, 1))
            for sv, (days, vol) in services.items():
                req = int(w * rng.uniform(800, 1500) * vol * (1 + 0.02 * m))
                proc = max(1.0, days * (1.35 - 0.7 * adoption) + rng.normal(0, 1.2))
                res = float(np.clip(62 + 34 * adoption - 0.4 * proc + rng.normal(0, 2.5), 40, 99))
                sat = float(np.clip(2.4 + 1.7 * adoption - 0.04 * proc + rng.normal(0, 0.2), 1, 5))
                label = str(rng.choice([st] * 4 + VARIANTS[st]))
                date = month_end.strftime("%Y-%m-%d") if rng.random() < 0.6 else month_end.strftime("%d/%m/%Y")
                unit, t = "days", round(proc, 2)
                if rng.random() < 0.12:
                    unit, t = "hours", round(proc * 24, 1)
                req_s = f"{req:,}" if rng.random() < 0.5 else str(req)
                res_s = f"{res:.1f}%" if rng.random() < 0.15 else f"{res:.1f}"
                rows.append([label, sv, date, req_s, t, unit, res_s, round(sat, 2)])
    cols = ["State/UT", "Service Name", "Reporting Date", "Requests Received",
            "Avg Processing Time", "Time Unit", "Resolved (%)", "Citizen Rating"]
    df = pd.DataFrame(rows, columns=cols).astype(object)
    n = len(df)
    pick = lambda k: rng.choice(n, k, replace=False)
    df.loc[pick(12), "Requests Received"] = "NA"
    df.loc[pick(18), "Avg Processing Time"] = "-"
    df.loc[pick(30), "Resolved (%)"] = ""
    df.loc[pick(35), "Citizen Rating"] = ""
    df.loc[pick(6), "Citizen Rating"] = 99          # sentinel value
    df.loc[pick(5), "Resolved (%)"] = "105.0"       # impossible percentage
    for i in pick(6):                               # extreme processing times (typing errors)
        v = df.loc[i, "Avg Processing Time"]
        if v != "-":
            df.loc[i, "Avg Processing Time"] = round(float(v) * 6, 2)
    df = pd.concat([df, df.sample(15, random_state=3)], ignore_index=True)   # duplicate rows
    return df

def simulate_source_b(seed=11):
    rng = np.random.default_rng(seed)
    base = {"Uttar Pradesh": (55, 58), "Maharashtra": (78, 74), "Tamil Nadu": (80, 78), "Karnataka": (79, 76),
            "Gujarat": (74, 72), "Rajasthan": (58, 60), "West Bengal": (62, 64), "Bihar": (48, 52),
            "Kerala": (88, 92), "Madhya Pradesh": (54, 57), "Odisha": (53, 58)}   # Telangana is missing on purpose
    rows = []
    for st, (subs, lit) in base.items():
        label = str(rng.choice([st] + VARIANTS[st]))
        rows.append([label, round(subs + rng.normal(0, 1.5), 1), round(lit + rng.normal(0, 1.5), 1)])
    rows.append(["Goa", 90.2, 88.4])    # a state that is not in Source A
    df = pd.DataFrame(rows, columns=["State", "Internet Subscribers per 100 Population", "Digital Literacy (%)"]).astype(object)
    df.loc[3, "Digital Literacy (%)"] = "NA"
    return df

PATH_A = "data/raw/service_performance_raw.csv"
PATH_B = "data/raw/digital_readiness_raw.csv"
simulate_source_a().to_csv(PATH_A, index=False)
simulate_source_b().to_csv(PATH_B, index=False)

def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:16]

# Read everything as text, exactly as downloaded (no silent type guessing)
raw_a = pd.read_csv(PATH_A, dtype=str, keep_default_na=False)
raw_b = pd.read_csv(PATH_B, dtype=str, keep_default_na=False)

metadata_log = pd.DataFrame([
    {"dataset_id": "DS-A", "name": "Service performance extract", "source_type": "OGD-style state service data (simulated)",
     "file": PATH_A, "format": "CSV", "date_accessed": datetime.date.today().isoformat(),
     "rows": len(raw_a), "columns": raw_a.shape[1], "checksum": sha256(PATH_A)},
    {"dataset_id": "DS-B", "name": "Digital readiness indicators", "source_type": "Telecom/survey-style state data (simulated)",
     "file": PATH_B, "format": "CSV", "date_accessed": datetime.date.today().isoformat(),
     "rows": len(raw_b), "columns": raw_b.shape[1], "checksum": sha256(PATH_B)},
])
metadata_log.to_csv("logs/metadata_log.csv", index=False)
metadata_log

## 3. Parse and standardise (types, names, dates, units)

In [ ]:
NA_TOKENS = {"", "na", "n/a", "-", "null", "none", "nan"}

def snake(c):
    return re.sub(r"[^0-9a-z]+", "_", c.strip().lower()).strip("_")

def to_num(s):
    """Text -> number: strip spaces, NA tokens to NaN, remove commas and % signs."""
    s = s.astype(str).str.strip().str.lower()
    s = s.where(~s.isin(NA_TOKENS), np.nan)
    s = s.str.replace(",", "", regex=False).str.replace("%", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def parse_dates(s):
    s = s.str.strip()
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return iso.combine_first(dmy)

# ---- Source A ----
a = raw_a.copy()
a.columns = [snake(c) for c in a.columns]
a = a.rename(columns={"state_ut": "state", "service_name": "service", "reporting_date": "date",
                      "requests_received": "requests", "avg_processing_time": "processing_time",
                      "resolved": "resolution_rate", "citizen_rating": "satisfaction"})
for c in ["state", "service", "time_unit"]:
    a[c] = a[c].str.strip()
a["time_unit"] = a["time_unit"].str.lower()
a["state_raw"] = a["state"]
a["state"] = a["state"].map(lambda s: STATE_MAP.get(norm_label(s)))
a["month"] = parse_dates(a["date"]).dt.strftime("%Y-%m")
for c in ["requests", "processing_time", "resolution_rate", "satisfaction"]:
    a[c] = to_num(a[c])
typed = a.drop(columns=["date"]).copy()    # typed but NOT yet cleaned: used for the "before" quality check

# ---- Source B ----
b = raw_b.copy()
b.columns = ["state", "internet_per_100", "digital_literacy_pct"]
b["state_raw"] = b["state"]
b["state"] = b["state"].map(lambda s: STATE_MAP.get(norm_label(s)))
for c in ["internet_per_100", "digital_literacy_pct"]:
    b[c] = to_num(b[c])

print("Source A typed:", typed.shape, "| unmapped state labels:", int(typed["state"].isna().sum()))
print("Source B typed:", b.shape, "| unmapped state labels:", int(b["state"].isna().sum()), "(not in the 12 study states)")
typed.head()

## 4. Validation rule set (used before and after cleaning)

Validation is written as reusable rules so the same checks run at each stage. Each rule returns the number of rows that **fail**.

In [ ]:
KEY = ["state", "service", "month"]
MEASURES = ["requests", "processing_days", "resolution_rate", "satisfaction"]

def run_checks(df, stage):
    d = df.copy()
    if "processing_days" not in d:
        d["processing_days"] = d["processing_time"]
    out = []
    def add(rid, rule, failed):
        out.append({"stage": stage, "rule_id": rid, "rule": rule, "failed_rows": int(failed),
                    "status": "PASS" if int(failed) == 0 else "FAIL"})
    add("R1", "Key columns (state, service, month) are complete", d[KEY].isna().any(axis=1).sum())
    add("R2", "Measure columns have no missing values", d[MEASURES].isna().any(axis=1).sum())
    add("R3", "Each state-service-month appears once (unique key)", d.duplicated(KEY).sum())
    add("R4", "Requests are zero or positive", (d["requests"] < 0).sum())
    add("R5", "Processing time is greater than zero", (d["processing_days"] <= 0).sum())
    add("R6", "Resolution rate is between 0 and 100", (~d["resolution_rate"].between(0, 100) & d["resolution_rate"].notna()).sum())
    add("R7", "Satisfaction is between 1 and 5", (~d["satisfaction"].between(1, 5) & d["satisfaction"].notna()).sum())
    add("R8", "Processing time is recorded in days (single unit)", (d["time_unit"] != "days").sum() if "time_unit" in d else 0)
    add("R9", "Month is inside the study period 2025-01 to 2025-12", (~d["month"].between("2025-01", "2025-12")).sum())
    add("R10", "State is one of the 12 study states", (~d["state"].isin(STATES)).sum())
    return pd.DataFrame(out)

checks_before = run_checks(typed, "before cleaning")
checks_before

## 5. Clean the data

Order matters. The steps below run in this order: unit harmonisation, invalid values to missing, duplicate removal, outlier capping, missing-value flags, then imputation. Each step is counted so the effect is documented in a cleaning log.

In [ ]:
log = []
def note(step, issue, n, action):
    log.append({"step": step, "issue": issue, "rows_or_cells_affected": int(n), "action": action})

c = typed.copy()
note("Standardise", "State name variants (case, dots, spaces, old names)", (c["state_raw"].str.strip() != c["state"]).sum(), "Mapped to standard names")

# 1) Harmonise units (hours -> days)
hours = c["time_unit"].eq("hours")
c["processing_days"] = np.where(hours, c["processing_time"] / 24, c["processing_time"])
note("Units", "Processing time recorded in hours", hours.sum(), "Converted to days (divide by 24)")

# 2) Invalid values -> missing (they are errors, not real measurements)
bad_res = ~c["resolution_rate"].between(0, 100) & c["resolution_rate"].notna()
bad_sat = ~c["satisfaction"].between(1, 5) & c["satisfaction"].notna()
c.loc[bad_res, "resolution_rate"] = np.nan
c.loc[bad_sat, "satisfaction"] = np.nan
note("Validity", "Resolution rate outside 0-100", bad_res.sum(), "Set to missing, then imputed")
note("Validity", "Satisfaction outside 1-5 (sentinel 99)", bad_sat.sum(), "Set to missing, then imputed")

# 3) Remove duplicate records (same state, service, month)
dups = c.duplicated(KEY).sum()
c = c.drop_duplicates(KEY, keep="first").copy()
note("Duplicates", "Repeated state-service-month records", dups, "Kept first occurrence")

# 4) Outliers: IQR rule per service on processing days, then cap (winsorise)
def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)

pre_cap = c[["service", "processing_days"]].copy()
n_out = 0
for sv, g in c.groupby("service"):
    lo, hi = iqr_bounds(g["processing_days"])
    mask = (c["service"] == sv) & ((c["processing_days"] < lo) | (c["processing_days"] > hi))
    n_out += int(mask.sum())
    c.loc[mask, "processing_days"] = c.loc[mask, "processing_days"].clip(lo, hi)
note("Outliers", "Extreme processing times (IQR rule per service)", n_out, "Capped at the IQR bounds")

# 5) Missing values: flag first, then impute
for col in MEASURES:
    c[col + "_was_missing"] = c[col].isna()
missing_cells = int(c[MEASURES].isna().sum().sum())
for col in ["requests", "processing_days"]:                      # group median by service
    c[col] = c.groupby("service")[col].transform(lambda s: s.fillna(s.median()))
sc = StandardScaler()
X = sc.fit_transform(c[MEASURES])
X_imp = KNNImputer(n_neighbors=5).fit_transform(X)               # KNN for rating and resolution
c[MEASURES] = sc.inverse_transform(X_imp)
c["resolution_rate"] = c["resolution_rate"].clip(0, 100).round(1)
c["satisfaction"] = c["satisfaction"].clip(1, 5).round(2)
c["processing_days"] = c["processing_days"].round(2)
c["requests"] = c["requests"].round(0).astype(int)
note("Missing", "Missing or invalid measure values", missing_cells, "Group median (requests, processing) and KNN (resolution, satisfaction); flag columns kept")

clean_a = c.drop(columns=["processing_time", "time_unit", "state_raw"]).reset_index(drop=True)
clean_a.head()

## 6. Integrate Source B and engineer features

In [ ]:
# Source B: drop states outside the study, impute the missing literacy value, then left-join
b_in = b[b["state"].notna()].drop(columns=["state_raw"]).copy()
note("Integration", "Source B rows for states outside the study (e.g. Goa)", int(b["state"].isna().sum()), "Dropped from the join")
b_missing = int(b_in[["internet_per_100", "digital_literacy_pct"]].isna().sum().sum())
for col in ["internet_per_100", "digital_literacy_pct"]:
    b_in[col] = b_in[col].fillna(b_in[col].median())
note("Missing", "Missing digital readiness value in Source B", b_missing, "Filled with the median of other states")

merged = clean_a.merge(b_in, on="state", how="left")
merged["readiness_imputed"] = merged["internet_per_100"].isna()
n_no_match = int(merged["readiness_imputed"].sum())
for col in ["internet_per_100", "digital_literacy_pct"]:
    merged[col] = merged[col].fillna(b_in[col].median())
note("Integration", "Study-state rows with no match in Source B (Telangana)", n_no_match, "Median readiness used and flagged in readiness_imputed")

# Feature engineering
mm = lambda s: (s - s.min()) / (s.max() - s.min())
merged["digital_adoption_index"] = ((mm(merged["internet_per_100"]) + mm(merged["digital_literacy_pct"])) / 2).round(3)
merged["quarter"] = "Q" + ((merged["month"].str[5:7].astype(int) - 1) // 3 + 1).astype(str)
merged["requests_log"] = np.log1p(merged["requests"])
merged["speed_category"] = pd.cut(merged["processing_days"], [0, 6, 12, np.inf], labels=["fast", "medium", "slow"])
merged["resolution_gap"] = (100 - merged["resolution_rate"]).round(1)

# Scaling (kept as separate columns) and one-hot encoding of service
scale_cols = ["requests_log", "processing_days", "resolution_rate", "satisfaction", "digital_adoption_index"]
z = StandardScaler().fit_transform(merged[scale_cols])
m = MinMaxScaler().fit_transform(merged[scale_cols])
analysis_ready = merged.copy()
for i, col in enumerate(scale_cols):
    analysis_ready[col + "_z"] = z[:, i].round(4)
    analysis_ready[col + "_mm"] = m[:, i].round(4)
analysis_ready = pd.concat([analysis_ready, pd.get_dummies(analysis_ready["service"], prefix="svc", dtype=int)], axis=1)

merged.to_csv("data/processed/cleaned_service_data.csv", index=False)
analysis_ready.to_csv("data/processed/analysis_ready.csv", index=False)
pd.DataFrame(log).to_csv("logs/cleaning_log.csv", index=False)
print("Cleaned:", merged.shape, "| Analysis-ready:", analysis_ready.shape)
analysis_ready.head()

## 7. Validate again and build the quality report

In [ ]:
checks_after = run_checks(merged, "after cleaning")
validation = pd.concat([checks_before, checks_after], ignore_index=True)
validation.to_csv("logs/validation_report.csv", index=False)

pivot = validation.pivot(index="rule_id", columns="stage", values="failed_rows")
pivot = pivot.loc[[f"R{i}" for i in range(1, 11)]]
rule_text = checks_before.set_index("rule_id")["rule"]
pivot.insert(0, "rule", rule_text)
print(pivot.to_string())

logdf = pd.DataFrame(log)
def logn(step_issue_start):
    return int(logdf[logdf["issue"].str.startswith(step_issue_start)]["rows_or_cells_affected"].sum())

metrics = {
    "rows_raw_a": len(raw_a), "rows_clean": len(merged), "cols_clean": merged.shape[1],
    "cols_analysis_ready": analysis_ready.shape[1],
    "duplicates_removed": int(dups), "hours_converted": int(hours.sum()),
    "invalid_nulled": int(bad_res.sum() + bad_sat.sum()), "outliers_capped": int(n_out),
    "cells_imputed_a": int(missing_cells), "cells_imputed_b": int(b_missing),
    "rows_readiness_no_match": int(n_no_match),
    "state_variants_fixed": logn("State name variants"),
    "rules_total": 10,
    "rules_passed_before": int((checks_before["status"] == "PASS").sum()),
    "rules_passed_after": int((checks_after["status"] == "PASS").sum()),
    "failed_rows_before": int(checks_before["failed_rows"].sum()),
    "failed_rows_after": int(checks_after["failed_rows"].sum()),
    "checks_before": checks_before[["rule_id", "rule", "failed_rows", "status"]].to_dict("records"),
    "checks_after": checks_after[["rule_id", "failed_rows", "status"]].to_dict("records"),
    "metadata": metadata_log.to_dict("records"),
    "log": log,
}
with open("logs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=1)
metrics["rules_passed_before"], metrics["rules_passed_after"]

## 8. Charts

In [ ]:
chart_files = {}

# Chart 1: issues found and fixed
issues = {
    "State names standardised": metrics["state_variants_fixed"],
    "Hours converted to days": metrics["hours_converted"],
    "Invalid values removed": metrics["invalid_nulled"],
    "Duplicate rows removed": metrics["duplicates_removed"],
    "Outliers capped": metrics["outliers_capped"],
    "Missing or invalid cells imputed": metrics["cells_imputed_a"] + metrics["cells_imputed_b"],
}
s = pd.Series(issues).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.barh(s.index, s.values, color="#1F4E79")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center")
ax.set_xlabel("Rows or cells affected")
ax.set_title("Data quality issues found and fixed")
plt.tight_layout()
chart_files["c1"] = "charts/01_issues_fixed.png"
plt.savefig(chart_files["c1"], dpi=110); plt.show(); plt.close()

# Chart 2: processing time before vs after
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].hist(typed["processing_time"].dropna(), bins=40, color="#C0392B")
axes[0].set_title("Before: mixed hours and days")
axes[1].hist(merged["processing_days"], bins=40, color="#2E8B57")
axes[1].set_title("After: days only, outliers capped")
for ax in axes:
    ax.set_xlabel("Processing time"); ax.set_ylabel("Records")
plt.tight_layout()
chart_files["c2"] = "charts/02_processing_before_after.png"
plt.savefig(chart_files["c2"], dpi=110); plt.show(); plt.close()

# Chart 3: outlier capping by service (boxplots)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4), sharey=True)
sns.boxplot(data=pre_cap, x="service", y="processing_days", ax=axes[0], color="#E8A09A")
axes[0].set_title("Before capping")
sns.boxplot(data=merged, x="service", y="processing_days", ax=axes[1], color="#9CCBA8")
axes[1].set_title("After capping")
for ax in axes:
    ax.set_xlabel(""); ax.set_ylabel("Processing days")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
chart_files["c3"] = "charts/03_outliers_by_service.png"
plt.savefig(chart_files["c3"], dpi=110); plt.show(); plt.close()

# Chart 4: effect of scaling
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
axes[0].hist(analysis_ready["processing_days"], bins=30, color="#7F7F7F"); axes[0].set_title("Original (days)")
axes[1].hist(analysis_ready["processing_days_z"], bins=30, color="#1F4E79"); axes[1].set_title("Standardised (z-score)")
axes[2].hist(analysis_ready["processing_days_mm"], bins=30, color="#2E8B57"); axes[2].set_title("Min-max (0 to 1)")
plt.tight_layout()
chart_files["c4"] = "charts/04_scaling_comparison.png"
plt.savefig(chart_files["c4"], dpi=110); plt.show(); plt.close()

# Chart 5: validation failures before vs after
pv = pivot.drop(columns="rule")
x = np.arange(len(pv)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(x - w / 2, pv["before cleaning"], w, label="Before cleaning", color="#C0392B")
ax.bar(x + w / 2, pv["after cleaning"], w, label="After cleaning", color="#2E8B57")
ax.set_xticks(x); ax.set_xticklabels(pv.index)
ax.set_ylabel("Failing rows"); ax.set_title("Validation rules: failing rows before and after cleaning")
ax.legend()
plt.tight_layout()
chart_files["c5"] = "charts/05_validation_before_after.png"
plt.savefig(chart_files["c5"], dpi=110); plt.show(); plt.close()

## 9. Export the HTML quality report

In [ ]:
import base64
def img64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

kpis = [
    ("Raw records (Source A)", f"{metrics['rows_raw_a']:,}"),
    ("Clean records", f"{metrics['rows_clean']:,}"),
    ("Validation rules passed", f"{metrics['rules_passed_before']} to {metrics['rules_passed_after']} of {metrics['rules_total']}"),
    ("Failing rows", f"{metrics['failed_rows_before']} to {metrics['failed_rows_after']}"),
    ("Analysis-ready columns", f"{metrics['cols_analysis_ready']}"),
]
kpi_html = "".join(f'<div class="kpi"><div class="v">{v}</div><div class="l">{l}</div></div>' for l, v in kpis)

M = metrics
sections = [
    ("c1", "Issues found and fixed",
     f"The raw extract had {M['state_variants_fixed']} rows with non-standard state names, {M['hours_converted']} processing times recorded in hours, {M['invalid_nulled']} impossible values, {M['duplicates_removed']} duplicate rows and {M['outliers_capped']} extreme outliers. {M['cells_imputed_a'] + M['cells_imputed_b']} missing or invalid cells (including the impossible values) were filled and flagged."),
    ("c2", "Processing time: before and after",
     "Mixing hours with days stretched the raw distribution into a long tail. After unit conversion and outlier capping the values sit on one consistent scale."),
    ("c3", "Outlier capping by service",
     f"Boxplots show extreme values per service before capping. {M['outliers_capped']} values were capped at the IQR bounds instead of being deleted, so no records were lost."),
    ("c4", "Effect of scaling",
     "Standardisation (mean 0, standard deviation 1) and min-max scaling (0 to 1) change the scale but not the shape of the distribution. Both versions are kept for later modelling."),
    ("c5", "Validation before and after",
     f"{M['rules_passed_before']} of {M['rules_total']} rules passed before cleaning and {M['rules_passed_after']} of {M['rules_total']} after. Failing rows fell from {M['failed_rows_before']} to {M['failed_rows_after']}."),
]
sec_html = "".join(
    f'<section><h2>{t}</h2><img src="data:image/png;base64,{img64(chart_files[k])}" alt="{t}"><p class="insight">{txt}</p></section>'
    for k, t, txt in sections
)

rows_html = "".join(
    f"<tr><td>{r['rule_id']}</td><td>{r['rule']}</td><td>{r['failed_rows']}</td><td>{M['checks_after'][i]['failed_rows']}</td></tr>"
    for i, r in enumerate(M["checks_before"])
)
val_html = ('<section><h2>Validation rule results</h2><table><tr><th>Rule</th><th>Description</th>'
            '<th>Failing rows before</th><th>Failing rows after</th></tr>' + rows_html + '</table></section>')

meta_rows = "".join(
    f"<tr><td>{r['dataset_id']}</td><td>{r['name']}</td><td>{r['format']}</td><td>{r['rows']}</td><td>{r['columns']}</td><td>{r['checksum']}</td></tr>"
    for r in M["metadata"]
)
meta_html = ('<section><h2>Collected files (metadata log)</h2><table><tr><th>ID</th><th>Name</th><th>Format</th>'
             '<th>Rows</th><th>Columns</th><th>Checksum</th></tr>' + meta_rows + '</table></section>')

TEMPLATE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Data Collection and Preprocessing: Quality Report</title>
<style>
body{font-family:Arial,Helvetica,sans-serif;margin:0;background:#f4f6f8;color:#1b2733}
header{background:#1F4E79;color:#fff;padding:28px 6vw}
header h1{margin:0 0 6px;font-size:26px}
header p{margin:0;opacity:.9}
main{max-width:980px;margin:0 auto;padding:20px 5vw 40px}
.note{background:#fff4d6;border-left:5px solid #e0a800;padding:12px 16px;margin:18px 0;border-radius:4px}
.kpis{display:flex;flex-wrap:wrap;gap:12px;margin:18px 0}
.kpi{flex:1 1 150px;background:#fff;border-radius:8px;padding:14px;box-shadow:0 1px 3px rgba(0,0,0,.12);text-align:center}
.kpi .v{font-size:20px;font-weight:bold;color:#1F4E79}
.kpi .l{font-size:12px;color:#5b6b79;margin-top:4px}
section{background:#fff;border-radius:8px;padding:18px;margin:18px 0;box-shadow:0 1px 3px rgba(0,0,0,.12);overflow-x:auto}
section h2{margin-top:0;font-size:19px;color:#1F4E79}
img{max-width:100%;height:auto;display:block;margin:0 auto}
.insight{background:#eef4fa;padding:10px 14px;border-radius:6px;margin-bottom:0}
table{border-collapse:collapse;width:100%;font-size:13px}
th{background:#1F4E79;color:#fff;text-align:left}
th,td{border:1px solid #cfd6dd;padding:7px 9px}
footer{text-align:center;color:#5b6b79;font-size:12px;padding:20px}
</style></head><body>
<header><h1>Data Collection and Preprocessing: Quality Report</h1>
<p>Week 2 prototype: raw data to analysis-ready data</p></header>
<main>
<div class="note"><strong>Data note:</strong> the raw files are simulated to look like messy public data. Numbers describe the cleaning method only and are not real government statistics.</div>
<div class="kpis">%%KPIS%%</div>
%%META%%
%%SECTIONS%%
%%VAL%%
</main>
<footer>Data analytics internship project | pandas, NumPy, scikit-learn, matplotlib, seaborn</footer>
</body></html>"""

html = (TEMPLATE.replace("%%KPIS%%", kpi_html).replace("%%META%%", meta_html)
        .replace("%%SECTIONS%%", sec_html).replace("%%VAL%%", val_html))
with open("egov_quality_report.html", "w", encoding="utf-8") as f:
    f.write(html)
print("Saved egov_quality_report.html")

## 10. Next steps

- Replace the simulated files in `data/raw/` with real public extracts (same columns) and re-run.
- Extend the state name mapping as new spellings appear (the cleaning log lists unmapped labels).
- Add the real validation thresholds agreed with the project mentor.
- Move to Week 3: exploratory analysis on `data/processed/analysis_ready.csv`.